In [27]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_val_score, KFold
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel

from scipy.stats import norm
from scipy.spatial.distance import cdist
from scipy.stats.qmc import LatinHypercube

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


# Helper Functions

In [29]:
def compute_ucb(mu, sigma, kappa=2.5):
    return mu + kappa * sigma

def compute_ei(mu, sigma, f_best, xi=0.01):
    z = (mu - f_best - xi) / (sigma + 1e-9)
    ei = (mu - f_best - xi) * norm.cdf(z) + sigma * norm.pdf(z)
    ei[sigma == 0] = 0
    return ei

def format_query(point, decimals=6):
    point_clipped = np.clip(point, 0, 0.999999)
    return '-'.join([f'{x:.{decimals}f}' for x in point_clipped])

def thompson_sample(gp, candidates, n_samples=10, subsample=500):
    idx = np.random.choice(len(candidates), size=min(subsample, len(candidates)), replace=False)
    sub = candidates[idx]
    samples = gp.sample_y(sub, n_samples=n_samples, random_state=42)
    best_sub_idx = np.argmax(samples.mean(axis=1))
    return idx[best_sub_idx]

# Function 1 - Week 5

In [31]:
# =============================================================================
# FUNCTION 1 - Radiation Detection (2D)
# changes: 
#     - fix length scale at 0.15 (week 4 length scale collapsed at 0.00375)
# =============================================================================

print("=" * 60)
print("Function 1 - Week 5")
print("=" * 60)

inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_1/initial_inputs.npy')
outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_1/initial_outputs.npy')

prev_queries = np.array([
    [0.959184, 0.836735],
    [0.374540, 0.950714],
    [0.694651, 0.629916],
    [0.461537, 0.459084],
])
prev_outputs = np.array([
    -5.909566597235814e-107,
    -1.560646704467778e-117,
    -0.0016067678433140744,
    -0.000016877758079573665,
])

all_inputs  = np.vstack([inputs, prev_queries])
all_outputs = np.hstack([outputs, prev_outputs])

print(f"Total points: {len(all_outputs)}")
print(f"Current best: {all_outputs.max():.6e}")

abs_outputs       = np.abs(all_outputs)           # all outputs are negative, log needs positive values
abs_outputs_floor = abs_outputs + 1e-300          # add tiny floor so log(0) = -inf never occurs
log_outputs       = np.log10(abs_outputs_floor)   # compress to roughly [-120, -2] range

# fix length scale at 0.15 -- estimated from pairwise distances between
# non-zero points (0.068 to 0.291), prevents optimiser collapse in week 4
kernel = Matern(nu=2.5, length_scale=0.15, length_scale_bounds="fixed")
gp     = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True)
gp.fit(all_inputs, log_outputs)

np.random.seed(42)
candidates = np.random.uniform(0, 1, (10000, 2))

mu, sigma = gp.predict(candidates, return_std=True)
ucb       = compute_ucb(mu, sigma, kappa=4.0)

best_idx = np.argmax(ucb)
query    = candidates[best_idx]

print(f"\nWeek 5 Query: {format_query(query)}")
print(f"mu: {mu[best_idx]:.4f}, sigma: {sigma[best_idx]:.4f}")

Function 1 - Week 5
Total points: 14
Current best: 7.710875e-16

Week 5 Query: 0.618043-0.460066
mu: -24.9386, sigma: 31.9769


# Function 2 - Week 5

In [32]:
# =============================================================================
# FUNCTION 2 - Noisy Log-Likelihood (2D)
# w4 output: 0.675 (new best) -- anisotropic kernel revealed x2 irrelevant
# changes: 
#      - switch to Random Forest 
#      - candidates: uniform random across full space, RF scores, take argmax
# =============================================================================

print("\n" + "=" * 60)
print("Function 2 - Week 5")
print("=" * 60)

initial_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_2/initial_inputs.npy')
initial_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_2/initial_outputs.npy')

prev_queries = np.array([
    [0.775510, 0.959184],
    [0.683114, 0.932567],
    [0.794441, 0.256481],
    [0.706387, 0.952221],
])
prev_outputs = np.array([0.16576674, 0.56974583, 0.27313450, 0.67545988])

all_inputs  = np.vstack([initial_inputs, prev_queries])
all_outputs = np.hstack([initial_outputs, prev_outputs])

print(f"Total points: {len(all_outputs)}")
print(f"Current best: {all_outputs.max():.3f}")

# RF 
rf = RandomForestRegressor(n_estimators=500, max_depth=3, min_samples_leaf=2, random_state=42)
rf.fit(all_inputs, all_outputs)
print(f"Feature importances: x1={rf.feature_importances_[0]:.3f}, x2={rf.feature_importances_[1]:.3f}")

# uniform random candidates across full space
np.random.seed(42)
candidates = np.random.uniform(0, 1, (50000, 2))
preds      = rf.predict(candidates)
query      = candidates[np.argmax(preds)]

print(f"\nWeek 5 Query: {format_query(query)}")
print(f"Predicted mean: {preds.max():.3f}")


Function 2 - Week 5
Total points: 14
Current best: 0.675
Feature importances: x1=0.811, x2=0.189

Week 5 Query: 0.693183-0.938929
Predicted mean: 0.570


# Function 3 - Week 5

In [33]:
# =============================================================================
# FUNCTION 3 - Drug Discovery (3D)
# changes: 
#      - drop GP, switch to RF on polynomial features (to capture compound interactions: A*B, B*C etc.)
#      - global candidate search
# =============================================================================

print("\n" + "=" * 60)
print("Function 3 - Week 5")
print("=" * 60)

f3_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_3/initial_inputs.npy')
f3_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_3/initial_outputs.npy')

prev_queries = np.array([
    [0.378956, 0.302768, 0.459346],
    [0.315339, 0.088659, 0.415174],
    [0.392735, 0.504381, 0.464332],
    [0.498607, 0.467046, 0.477827],
])
prev_outputs = np.array([
    -0.02906213067759293,
    -0.06230989251412482,
    -0.0006328364393800658,
    -0.0029446702316140265,
])

X = np.vstack([f3_inputs, prev_queries])
Y = np.hstack([f3_outputs, prev_outputs])

print(f"Total points: {len(Y)}, best: {Y.max():.6f}")

# --- Polynomial features (explicit compound interactions) ---
poly   = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

feature_names = ['A', 'B', 'C', 'A*B', 'A*C', 'B*C', 'A^2', 'B^2', 'C^2']

# --- RF on polynomial features ---
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_poly, Y)

print("\nRF feature importances:")
for name, imp in zip(feature_names, rf.feature_importances_):
    print(f"  {name:6s}: {imp:.4f}")

# --- Global candidate search ---
np.random.seed(42)
candidates     = np.random.uniform(0, 1, (10000, 3))
cand_poly      = poly.transform(candidates)
rf_preds       = rf.predict(cand_poly)

best_idx = np.argmax(rf_preds)
query    = candidates[best_idx]

print(f"\nWeek 5 Query: {format_query(query)}")
print(f"RF predicted mean: {rf_preds[best_idx]:.6f}")


Function 3 - Week 5
Total points: 19, best: -0.000633

RF feature importances:
  A     : 0.0495
  B     : 0.0273
  C     : 0.3410
  A*B   : 0.0551
  A*C   : 0.0168
  B*C   : 0.0685
  A^2   : 0.0315
  B^2   : 0.1049
  C^2   : 0.3055

Week 5 Query: 0.432840-0.535542-0.476983
RF predicted mean: -0.009568


# Function 4 - Week 5

In [35]:
# =============================================================================
# FUNCTION 4 - Warehouse Placement (4D)
# changes: 
#    - RF surrogate + structured local search around w2
#    - pinned dimensions: x3 tight and everything else slightly looser
# =============================================================================

print("\n" + "=" * 60)
print("Function 4 - Week 5")
print("=" * 60)

f4_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_4/initial_inputs.npy')
f4_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_4/initial_outputs.npy')

prev_queries = np.array([
    [0.466173, 0.451984, 0.359193, 0.383111],
    [0.424201, 0.406375, 0.372722, 0.413313],
    [0.413541, 0.373697, 0.536536, 0.453354],
    [0.424125, 0.404716, 0.487507, 0.367688],
])
prev_outputs = np.array([-0.9654345395220925, 0.6308582112564989, -2.1500998298742817, -0.9915950770116662])

all_inputs  = np.vstack([f4_inputs, prev_queries])
all_outputs = np.hstack([f4_outputs, prev_outputs])

print(f"Total points: {len(all_outputs)}, best: {all_outputs.max():.4f}")

rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=2,
                           max_features='sqrt', random_state=42)
rf.fit(all_inputs, all_outputs)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, all_inputs, all_outputs, cv=kfold, scoring='r2')
print(f"CV: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"Feature importances: {rf.feature_importances_.round(4)}")

def rf_predict_with_std(rf, X):
    tree_preds = np.array([t.predict(X) for t in rf.estimators_])
    return tree_preds.mean(axis=0), tree_preds.std(axis=0)

candidates = LatinHypercube(d=4, seed=42).random(n=20000)

mu, sigma = rf_predict_with_std(rf, candidates)
best_y    = all_outputs.max()
z  = (mu - best_y - 0.01) / (sigma + 1e-9)
ei = (mu - best_y - 0.01) * norm.cdf(z) + sigma * norm.pdf(z)

query = candidates[np.argmax(ei)]

mu_q, sigma_q = rf_predict_with_std(rf, query.reshape(1, -1))

print(f"\nWeek 5 Query: {format_query(query)}")
print(f"Predicted mean: {mu_q[0]:.4f}, std: {sigma_q[0]:.4f}")
print(f"Distance from W2: {np.linalg.norm(query - w2):.4f}")
print(f"x3 delta from W2: {query[2] - w2[2]:+.4f}")


Function 4 - Week 5
Total points: 34, best: 0.6309
CV: 0.6492 (+/- 0.0663)
Feature importances: [0.3102 0.3344 0.1165 0.2389]

Week 5 Query: 0.488793-0.294975-0.332280-0.395576
Predicted mean: -5.1005, std: 6.2503
Distance from W2: 0.1361
x3 delta from W2: -0.0404


# Function 5 - Week 5

In [37]:
# =============================================================================
# FUNCTION 5 - Chemical Yield (4D)
# changes: 
#     - log-transform outputs for GP, removed LR (misleading on x1)
#     - SVR pre-filter and directed candidates kept unchanged from w4
#     - x3 pinned at 0.999999 (saturated since w1)
# =============================================================================

print("\n" + "=" * 60)
print("Function 5 - Week 5")
print("=" * 60)

f5_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_5/initial_inputs.npy')
f5_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_5/initial_outputs.npy')

prev_queries = np.array([
    [0.284290, 0.869208, 0.999999, 0.903273],  # w1: 2201.83
    [0.311208, 0.881577, 0.999999, 0.915050],  # w2: 2381.54
    [0.355085, 0.899160, 0.999900, 0.934196],  # w3: 2689.15
    [0.453609, 0.922659, 0.999999, 0.960267],  # w4: 3223.24
])
prev_outputs = np.array([2201.834589108927, 2381.536867607932, 2689.1537294933396, 3223.2410694936825])

all_inputs  = np.vstack([f5_inputs, prev_queries])
all_outputs = np.hstack([f5_outputs, prev_outputs])

print(f"Total points: {len(all_outputs)}, best so far: {all_outputs.max():.2f}")

# SVR: StandardScaler on raw outputs (unchanged from w4)
input_scaler  = StandardScaler()
output_scaler = StandardScaler()
X_scaled = input_scaler.fit_transform(all_inputs)
Y_scaled = output_scaler.fit_transform(all_outputs.reshape(-1, 1)).ravel()

svr = SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1)
svr.fit(X_scaled, Y_scaled)

# GP: log-transform outputs so all 24 points contribute equally
# raw range 0.11 -> 3223 causes GP to underweight original 20 points
y_log     = np.log1p(all_outputs)
kernel_gp = ConstantKernel(1.0, (0.001, 1000)) * Matern(length_scale=0.2, nu=2.5)
gp        = GaussianProcessRegressor(kernel=kernel_gp, n_restarts_optimizer=25, normalize_y=True)
gp.fit(all_inputs, y_log)

best_point = all_inputs[np.argmax(all_outputs)]
np.random.seed(42)

# local: small perturbations around w4 best (unchanged from w4)
local = []
for _ in range(6000):
    c    = best_point + np.random.normal(0, 0.03, 4)
    c    = np.clip(c, 0, 1)
    c[2] = 0.999999  # x3 pinned
    local.append(c)

# directed: x1 down, x2 and x4 up (unchanged from w4)
directed = []
for _ in range(4000):
    c    = best_point.copy()
    c[0] = np.clip(c[0] - abs(np.random.normal(0, 0.05)), 0, 1)
    c[1] = np.clip(c[1] + abs(np.random.normal(0, 0.03)), 0, 1)
    c[2] = 0.999999
    c[3] = np.clip(c[3] + abs(np.random.normal(0, 0.03)), 0, 1)
    directed.append(c)

candidates = np.vstack([local, directed])

# SVR pre-filter: keep top 3000 by predicted yield (unchanged from w4)
X_cand_scaled = input_scaler.transform(candidates)
svr_preds_s   = svr.predict(X_cand_scaled)
svr_preds     = output_scaler.inverse_transform(svr_preds_s.reshape(-1, 1)).ravel()

top_idx  = np.argsort(svr_preds)[-3000:]
filtered = candidates[top_idx]

# GP + EI on log scale
mu_log, sigma_log = gp.predict(filtered, return_std=True)
ei                = compute_ei(mu_log, sigma_log, np.log1p(all_outputs.max()), xi=0.005)

best_idx = np.argmax(ei)
query    = filtered[best_idx]

print(f"\nWeek 5 Query: {format_query(query)}")
print(f"GP predicted yield: {np.expm1(mu_log[best_idx]):.2f}")
print(f"GP std (log scale): {sigma_log[best_idx]:.4f}")
print(f"SVR prediction: {svr_preds[top_idx[best_idx]]:.2f}")
print(f"EI: {ei[best_idx]:.6f}")


Function 5 - Week 5
Total points: 24, best so far: 3223.24

Week 5 Query: 0.493503-0.872392-0.999999-0.999999
GP predicted yield: 3941.06
GP std (log scale): 0.3130
SVR prediction: 3139.37
EI: 0.246579


# Function 6 - Week 5

In [39]:
# =============================================================================
# FUNCTION 6 - Cake Recipe (5D)
# change: 
#    - replacing GP with a Random Forest surrogate
#    - acquisition: thompson sampling via a single random tree to drive exploration
# =============================================================================

print("\n" + "=" * 60)
print("Function 6 - Week 5")
print("=" * 60)

f6_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_6/initial_inputs.npy')
f6_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_6/initial_outputs.npy')

prev_queries = np.array([
    [0.201818, 0.220639, 0.476414, 0.867793, 0.023115],
    [0.457382, 0.349799, 0.529520, 0.691032, 0.192589],
    [0.503404, 0.328861, 0.662838, 0.995031, 0.025514],
    [0.484266, 0.300747, 0.678836, 0.776480, 0.244496],
])
prev_outputs = np.array([-0.792246, -0.361637, -0.367626, -0.314817])

all_inputs  = np.vstack([f6_inputs, prev_queries])
all_outputs = np.hstack([f6_outputs, prev_outputs])

print(f"Total observations: {all_inputs.shape[0]}, best: {all_outputs.max():.4f}")

# Random Forest surrogate
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(all_inputs, all_outputs)

print("\nRandom Forest feature importances:")
for i, importance in enumerate(rf.feature_importances_):
    print(f"  x{i+1}: {importance:.4f}")

# Candidate generation - random uniform across full space
np.random.seed(42)
candidates = np.random.uniform(0, 1, (15000, 5))

# Thompson sampling - sample a single random tree to encourage exploration
# individual trees disagree most in unexplored regions, trying to naturally drive search there
np.random.seed(42)
random_tree    = rf.estimators_[np.random.randint(0, len(rf.estimators_))]
tree_predicted = random_tree.predict(candidates)
best_index     = np.argmax(tree_predicted)
query          = candidates[best_index]

print(f"\nWeek 5 Query: {format_query(query)}")
print(f"Tree predicted mean: {tree_predicted[best_index]:.4f}")


Function 6 - Week 5
Total observations: 24, best: -0.3148

Random Forest feature importances:
  x1: 0.0702
  x2: 0.1285
  x3: 0.0630
  x4: 0.4173
  x5: 0.3210

Week 5 Query: 0.341066-0.113474-0.924694-0.877339-0.257942
Tree predicted mean: -0.3148


# Function 7 - Week 5

In [40]:
# =============================================================================
# FUNCTION 7 - ML Hyperparameters (6D)
# changes: 
#     - replace GP with RBF (multiquadric) + RF
#          step 1: RBF scores full space first - can extrapolate beyond best seen
#          step 2: RF validates RBF's top candidates - data-driven reality check
#     - no bounds constraints - function description suggests multi-modal landscape
# =============================================================================

print("\n" + "=" * 60)
print("Function 7 - Week 5")
print("=" * 60)

f7_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_7/initial_inputs.npy')
f7_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_7/initial_outputs.npy')

prev_queries = np.array([
    [0.045091, 0.528666, 0.329265, 0.105350, 0.434671, 0.641164],  # w1
    [0.034389, 0.427542, 0.236649, 0.342956, 0.405706, 0.695527],  # w2
    [0.020000, 0.062635, 0.587368, 0.313225, 0.362008, 0.664539],  # w3
    [0.012096, 0.033240, 0.640160, 0.336533, 0.245422, 0.901043],  # w4
])
prev_outputs = np.array([1.0510006614196026, 1.6531363312716738, 2.6016443512251484, 1.5087286481808686])

all_inputs  = np.vstack([f7_inputs, prev_queries])
all_outputs = np.hstack([f7_outputs, prev_outputs])
log_outputs = np.log1p(all_outputs)

print(f"Total points: {len(all_outputs)}, best: {all_outputs.max():.4f}")

# RF importance and directional signals
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(all_inputs, log_outputs)

feature_names = [f"HP{i+1}" for i in range(6)]
print("\nRF feature importance:")
for name, imp in zip(feature_names, rf.feature_importances_):
    print(f"  {name}: {imp:.3f}")


# RBF with multiquadric kernel 
rbf = RBFInterpolator(all_inputs, log_outputs, kernel='multiquadric', epsilon=1.0)

# candidates: full space no constraints, function is likely multi-modal
np.random.seed(42)
n          = 50000
candidates = np.random.uniform(0, 1, (n, 6))

# step 1: RBF scores all candidates - can go beyond best seen (2.60)
rbf_preds  = rbf(candidates)

# step 2: RF validates RBF's top candidates - checks data support
top_n      = 1000
top_rbf    = np.argsort(rbf_preds)[-top_n:]
rf_preds   = rf.predict(candidates[top_rbf])

# step 3: pick highest RF score within RBF's top candidates
best_idx   = top_rbf[np.argmax(rf_preds)]
query      = candidates[best_idx]

print(f"\nWeek 5 Query: {format_query(query)}")
print(f"RBF prediction (original): {np.expm1(rbf_preds[best_idx]):.4f}")
print(f"RF  prediction (original): {np.expm1(rf_preds[np.argmax(rf_preds)]):.4f}")


Function 7 - Week 5
Total points: 34, best: 2.6016

RF feature importance:
  HP1: 0.720
  HP2: 0.057
  HP3: 0.035
  HP4: 0.024
  HP5: 0.088
  HP6: 0.077

Week 5 Query: 0.007872-0.174935-0.499616-0.265730-0.340533-0.150339
RBF prediction (original): 1.1894
RF  prediction (original): 2.0142


# Function 8 - Week 5

In [41]:
# =============================================================================
# FUNCTION 8 - 8D Black-Box
# changes: 
#     - switch to MC dropout MLP as primary surrogate, no pinning
#     - candidate search random uniform on full space 
#     - acquisition: MC dropout UCB (50 forward passes with dropout active)
# =============================================================================

print("\n" + "=" * 60)
print("Function 8 - Week 5")
print("=" * 60)

import torch
import torch.nn as nn

f8_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_8/initial_inputs.npy')
f8_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_8/initial_outputs.npy')

prev_queries = np.array([
    [0.182943, 0.000000, 0.000000, 0.000000, 0.999999, 0.127124, 0.092403, 0.000000],
    [0.022695, 0.000000, 0.274348, 0.000000, 0.999999, 0.127124, 0.028034, 0.000000],
    [0.005086, 0.000000, 0.048567, 0.000000, 0.999999, 0.998962, 0.018674, 0.000000],
    [0.000000, 0.000000, 0.012582, 0.000000, 0.999999, 0.000000, 0.066411, 0.000000],
])
prev_outputs = np.array([9.6723503773075, 9.6263579169495, 9.5463675507445, 9.5519471979855])

all_inputs  = np.vstack([f8_inputs, prev_queries])
all_outputs = np.hstack([f8_outputs, prev_outputs])

print(f"Total points: {len(all_outputs)}, best: {all_outputs.max():.4f} (W1)")

# scale inputs and outputs
scaler_x = StandardScaler()
scaler_y = StandardScaler()
X_sc = scaler_x.fit_transform(all_inputs)
Y_sc = scaler_y.fit_transform(all_outputs.reshape(-1, 1)).ravel()

X_tensor = torch.tensor(X_sc, dtype=torch.float32)
Y_tensor = torch.tensor(Y_sc, dtype=torch.float32).unsqueeze(1)

# simple MLP with dropout using pytorch
# dropout stays active during prediction for MC uncertainty estimates
class DropoutMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout_rate):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
model     = DropoutMLP(input_dim=8, hidden_dim=32, dropout_rate=0.1)
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.MSELoss()

# train
model.train()
for epoch in range(1000):
    optimiser.zero_grad()
    pred = model(X_tensor)
    loss = loss_fn(pred, Y_tensor)
    loss.backward()
    optimiser.step()

print(f"Training loss (final): {loss.item():.4f}")

# MC dropout prediction -- keep model in train() so dropout stays active
def mc_predict(model, X_tensor, n_passes=50):
    model.train()
    preds = torch.stack([model(X_tensor).squeeze() for _ in range(n_passes)])
    return preds.mean(dim=0).detach().numpy(), preds.std(dim=0).detach().numpy()

# full 8D candidate pool -- no pins
np.random.seed(42)
n    = 50000
pool = np.random.uniform(0, 1, (n, 8))

pool_sc     = scaler_x.transform(pool)
pool_tensor = torch.tensor(pool_sc, dtype=torch.float32)

mu_sc, sigma_sc = mc_predict(model, pool_tensor, n_passes=50)

# inverse transform mu back to original scale for interpretability
mu    = scaler_y.inverse_transform(mu_sc.reshape(-1, 1)).ravel()
sigma = sigma_sc * scaler_y.scale_[0]

# UCB acquisition
kappa = 3.0 
ucb   = mu + kappa * sigma
best_idx = np.argmax(ucb)
query    = pool[best_idx]

print(f"\nWeek 5 Query: {format_query(query)}")
print(f"MC mu: {mu[best_idx]:.4f}, sigma: {sigma[best_idx]:.4f}")
print(f"UCB:   {ucb[best_idx]:.4f}")


Function 8 - Week 5
Total points: 44, best: 9.6724 (W1)
Training loss (final): 0.0196

Week 5 Query: 0.143046-0.934478-0.256793-0.378273-0.985471-0.743394-0.090143-0.014829
MC mu: 9.6347, sigma: 0.3245
UCB:   10.6082
